# Tarea 2 – Parte 2: Evaluación y extracción estructurada

**Pregunta guía:** ¿qué tan fiel es el texto que producen el OCR y el ASR de la Parte 1, y se puede extraer de él información estructurada con un LLM local?

| Paso | Qué se hace | Sección |
| :--- | :--- | :--- |
| Muestra | 10 documentos y 10 clips del corpus de la Parte 1, con transcripción manual de referencia | 1 |
| Normalización | Un solo criterio para referencias y salidas automáticas | 2 |
| Línea base | CER de PaddleOCR y WER de faster-whisper `small` | 3 |
| Comparación | EasyOCR y Parakeet en las mismas muestras, y casos difíciles | 4 |
| LLM | Extracción en JSON de los 48 archivos con Qwen2.5 local y auditoría humana de 20 | 5 |
| Decisión | Tabla comparativa, modelo elegido y reflexión | 6 |

**Privacidad.** Igual que en la Parte 1, este notebook solo muestra textos seudonimizados y conteos. Las referencias manuales de las entrevistas tienen los nombres reales de los niños: se guardan en `datos/raw/referencias/` (fuera de Git) y se seudonimizan antes de escribirlas en los CSV. Las métricas se calculan sobre los textos sin seudonimizar, para que un nombre mal escrito por el ASR cuente como error. Solo se publican los números.

In [1]:
import gc
import json
import os
import random
import re
import time
import unicodedata
import warnings
from datetime import date
from importlib.metadata import version
from pathlib import Path

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")

import jiwer
import pandas as pd
import plotly.express as px
import plotly.io as pio
import psutil
import truststore
from rapidfuzz import fuzz

# Valida HTTPS con los certificados del sistema operativo (el antivirus intercepta HTTPS)
truststore.inject_into_ssl()

pio.renderers.default = "plotly_mimetype+png"
pio.renderers["png"].width = 950
pio.templates.default = "plotly_white"
pio.templates["plotly_white"].layout.separators = ", "   # coma decimal en los ejes
pd.set_option("display.max_colwidth", 110)

DATOS, MODELOS = Path("datos"), Path("modelos")
RAW = DATOS / "raw"
REF_DIR = RAW / "referencias"     # transcripciones manuales con nombres reales (fuera de Git)
EVAL_DIR = RAW / "evaluacion"     # salidas crudas de los modelos de esta parte (fuera de Git)
FECHA = date.today().isoformat()
PROC = psutil.Process()

docs = pd.read_csv(DATOS / "documentos.csv", keep_default_na=False)
audios = pd.read_csv(DATOS / "transcripciones.csv", keep_default_na=False)
print(f"Fecha de evaluación: {FECHA} · corpus de la Parte 1: {len(docs)} documentos y {len(audios)} clips")
print(" · ".join(f"{p} {version(p)}" for p in ["jiwer", "paddleocr", "easyocr", "faster-whisper", "onnx-asr", "llama_cpp_python"]))

Fecha de evaluación: 2026-10-25 · corpus de la Parte 1: 25 documentos y 23 clips
jiwer 4.0.0 · paddleocr 3.7.0 · easyocr 1.7.2 · faster-whisper 1.2.1 · onnx-asr 0.12.0 · llama_cpp_python 0.3.2


## 1. Muestra de evaluación

La muestra sale del corpus de la Parte 1, sin datos nuevos, y se eligió antes de correr cualquier modelo de esta parte.

- **10 documentos**, sorteados por tipo con una semilla fija (`random.Random(42)`): 3 impresos, 3 fotos de celular y 4 manuscritos. Los manuscritos llevan uno más porque son el caso más difícil y el más variado.
- **10 clips de dos fuentes con condiciones de grabación distintas:**
  - Las **5 entrevistas más cortas**: conversación espontánea entre la profesora y un niño de unos 5 años, grabada con el celular en el jardín. Se eligieron por duración porque cada minuto hay que transcribirlo a mano.
  - **5 frases de OpenSLR 73**, sorteadas con la misma semilla: lectura en voz alta de un adulto, grabada en silencio.
  - Los dos podcasts (8 y 20 min) quedan fuera por su duración.

In [2]:
rng = random.Random(42)
CUPO = {"impreso": 3, "foto": 3, "manuscrito": 4}
muestra_docs = [i for tipo, k in CUPO.items()
                for i in sorted(rng.sample(sorted(docs.loc[docs["tipo"] == tipo, "id"]), k))]
entrevistas = sorted(audios[audios["fuente"] == "Entrevista propia"].nsmallest(5, "duracion_seg")["id"])
lecturas = sorted(rng.sample(sorted(audios.loc[audios["fuente"] == "OpenSLR 73", "id"]), 5))
muestra_audios = entrevistas + lecturas

CONDICION = {"Entrevista propia": "entrevista (habla espontánea de niños, celular, aula)",
             "OpenSLR 73": "lectura (adulto, micrófono, silencio)"}
m_docs = docs.set_index("id").loc[muestra_docs, ["tipo"]]
m_audios = audios.set_index("id").loc[muestra_audios, ["fuente", "duracion_seg", "n_hablantes"]]
m_audios["condicion"] = m_audios["fuente"].map(CONDICION)
print(f"Documentos: {m_docs['tipo'].value_counts().to_dict()}")
print(f"Clips: {len(m_audios)} ({m_audios['duracion_seg'].sum() / 60:.1f} min)")
m_audios

Documentos: {'manuscrito': 4, 'impreso': 3, 'foto': 3}
Clips: 10 (8.9 min)


,fuente,duracion_seg,n_hablantes,condicion
id,,,,
clip_004,Entrevista propia,123.8,2,"entrevista (habla espontánea de niños, celular, aula)"
clip_007,Entrevista propia,86.1,2,"entrevista (habla espontánea de niños, celular, aula)"
clip_008,Entrevista propia,124.0,2,"entrevista (habla espontánea de niños, celular, aula)"
clip_009,Entrevista propia,116.6,2,"entrevista (habla espontánea de niños, celular, aula)"
clip_010,Entrevista propia,42.7,2,"entrevista (habla espontánea de niños, celular, aula)"
clip_012,OpenSLR 73,7.0,1,"lectura (adulto, micrófono, silencio)"
clip_013,OpenSLR 73,9.6,1,"lectura (adulto, micrófono, silencio)"
clip_018,OpenSLR 73,7.9,1,"lectura (adulto, micrófono, silencio)"
clip_020,OpenSLR 73,9.7,1,"lectura (adulto, micrófono, silencio)"


### 1.1 Transcripciones manuales de referencia

La estudiante transcribió a mano los 20 archivos: un `.txt` por archivo en `datos/raw/referencias/{id}.txt`, en UTF-8. El método fue distinto según el tipo:

| Archivos | Cómo se hizo la referencia |
| :--- | :--- |
| `impreso1`, `impreso2` | La capa de texto del PDF original de las bases (SEACE), que la estudiante comparó con la imagen y corrigió donde no coincidía |
| `impreso6` | Esa página del PDF es un escaneo: solo el encabezado viene de la capa de texto y el cuerpo se transcribió a mano |
| `foto2`, `foto5`, `foto7` | A mano, mirando la foto. En `foto5` y `foto7` se transparentan del reverso unos nombres espejados, que no se transcribieron: no son texto de la página y son datos personales |
| `manuscrito2`, `3`, `6`, `7` | A mano y de forma literal, sin modernizar la ortografía |
| 5 entrevistas | A mano y desde cero, escuchando el audio sin mirar el texto del ASR: una intervención por línea con la etiqueta `H1` o `H2`. Los nombres se escribieron como suenan |
| 5 frases de OpenSLR | De oído, sin mirar la transcripción oficial del corpus |

Las referencias se cargan sin tocar, y para los CSV se seudonimizan con la misma revisión manual de la Parte 1 (`datos/raw/seudonimizacion/revision.csv`). En esta parte se le agregaron:

- el nombre de una profesora que aparece en una referencia;
- tres formas de nombres de niños que escribieron los ASR alternativos;
- por decisión de la estudiante, el nombre del colegio que menciona un niño en `clip_010`, que pasa a `[INSTITUCION]`. Junto con el seudónimo del niño, el colegio podría ayudar a identificarlo. El ASR de la Parte 1 no lo había transcrito, así que nunca se publicó.

Al pasar a los CSV, las etiquetas `H1`/`H2` pasan al formato `[HABLANTE_1]`/`[HABLANTE_2]` de `texto_asr`. La numeración es la de la estudiante, no la de la diarización.

In [3]:
PATRONES = {
    "DNI": r"(?i)\bD\.?\s?N\.?\s?I\.?\s*(?:N[°ºo.]*\s*)?:?\s*\d{8}\b",
    "CELULAR": r"(?<!\d)9\d{2}[\s-]?\d{3}[\s-]?\d{3}(?!\d)",
    "CORREO": r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+",
    "RUC": r"(?<!\d)10\d{9}(?!\d)",
}
revision = pd.read_csv(RAW / "seudonimizacion" / "revision.csv", keep_default_na=False)
personas = revision[(revision["decision"] == "reemplazar") & revision["tipo"].isin(["PERSONA", "INSTITUCION"])]
MANTENER = set(revision.loc[revision["decision"] == "mantener", "entidad"])


def seudonimizar(texto, reemplazos, parecido=80):
    """La misma función de 01_extraccion.ipynb (sección 6): primero coincidencias exactas y después
    aproximadas (rapidfuzz ≥ 80), estas solo para nombres de 5 letras o más."""
    salida = texto
    for exacto in (True, False):
        for nombre, seudonimo in reemplazos.items():
            if not exacto and len(nombre) < 5:
                continue
            k = len(nombre.split())
            palabras = re.findall(r"\w+|\W+", salida)
            idx = [i for i, p in enumerate(palabras) if re.match(r"\w", p)]
            cambios, j = [], 0
            while j <= len(idx) - k:
                a, b = idx[j], idx[j + k - 1]
                tramo = "".join(palabras[a:b + 1]).lower()
                if (tramo == nombre.lower()) if exacto else (fuzz.ratio(tramo, nombre.lower()) >= parecido):
                    cambios.append((a, b))
                    j += k
                else:
                    j += 1
            for a, b in reversed(cambios):
                palabras[a:b + 1] = [seudonimo]
            salida = "".join(palabras)
    return salida


def nombres_para(ident):
    "Nombres cuyo ámbito incluye este archivo; primero los más largos."
    aplica = personas[[a == "todos" or ident in a.split(";") for a in personas["ambito"]]]
    return dict(sorted(zip(aplica["entidad"], aplica["seudonimo"]), key=lambda kv: -len(kv[0])))


def publicable(texto, ident):
    "Texto listo para publicar: datos numéricos y correos con su marca, nombres con su seudónimo."
    for tipo, patron in PATRONES.items():
        texto = re.sub(patron, lambda m: m.group() if m.group() in MANTENER else f"[{tipo}]", texto)
    return seudonimizar(texto, nombres_para(ident))


# Etiqueta de hablante de las referencias: al inicio de línea con «:» o tabulador, o «H1:» a mitad de línea
ETIQUETA = r"(?m)(?:^[ \t]*[Hh](\d)\d?[ \t]*[:\t]|(?<!\w)[Hh](\d)\d?[ \t]*:)"


def a_marcas(texto):
    "Etiquetas H1/H2 de la referencia -> [HABLANTE_1]/[HABLANTE_2]."
    texto = re.sub(ETIQUETA, lambda m: f" [HABLANTE_{m.group(1) or m.group(2)}] ", texto)
    return " ".join(texto.split())


referencias = {i: (REF_DIR / f"{i}.txt").read_text(encoding="utf-8") for i in muestra_docs + muestra_audios}
vacias = [i for i, t in referencias.items() if not t.strip()]
assert not vacias, f"Faltan referencias: {vacias}"

docs["texto_referencia"] = [publicable(referencias[i], i) if i in referencias else "" for i in docs["id"]]
audios["texto_referencia"] = [publicable(a_marcas(referencias[i]), i) if i in referencias else "" for i in audios["id"]]
docs.to_csv(DATOS / "documentos.csv", index=False)
audios.to_csv(DATOS / "transcripciones.csv", index=False)

# Comprobación: ningún nombre marcado para reemplazar sobrevive en las referencias publicadas
quedan = [(i, n) for d in (docs, audios) for i, t in zip(d["id"], d["texto_referencia"]) for n in nombres_para(i)
          if t and re.search(rf"\b{re.escape(n)}\b", t, re.I)]
tamano = pd.DataFrame({"id": list(referencias), "palabras": [len(t.split()) for t in referencias.values()],
                       "caracteres": [len(" ".join(t.split())) for t in referencias.values()]})
print(f"Referencias: {len(referencias)} · {tamano['palabras'].sum()} palabras · nombres que siguen en el texto: {len(quedan)}")
print(f"texto_referencia completo en documentos.csv: {(docs['texto_referencia'] != '').sum()} y "
      f"en transcripciones.csv: {(audios['texto_referencia'] != '').sum()}")
tamano.set_index("id").T

Referencias: 20 · 3909 palabras · nombres que siguen en el texto: 0
texto_referencia completo en documentos.csv: 10 y en transcripciones.csv: 10


id,impreso1,impreso2,impreso6,foto2,foto5,foto7,manuscrito2,manuscrito3,manuscrito6,manuscrito7,clip_004,clip_007,clip_008,clip_009,clip_010,clip_012,clip_013,clip_018,clip_020,clip_021
palabras,286,450,345,330,479,475,219,97,55,51,244,212,289,191,85,22,18,20,21,20
caracteres,1834,3035,2476,2250,2886,2957,1264,573,372,348,1317,1010,1470,1046,507,95,91,95,120,84


### 1.2 Configuraciones de los modelos

Todas las configuraciones quedan fijadas aquí, antes de calcular cualquier métrica de esta parte, y no se cambian después de ver los resultados. Se guardan en `modelos/configuracion_evaluacion.json`.

- **Línea base sin personalización.** PaddleOCR y faster-whisper usan sus modelos preentrenados, sin ajuste fino ni diccionarios propios. Las salidas de PaddleOCR son las de la Parte 1. Su único cambio respecto a los valores por defecto es un límite de 3000 px para el lado mayor de la imagen, que se puso por memoria (con 4000 px los manuscritos grandes agotaban los 7 GB de RAM) y no por precisión.
- **ASR en igualdad de condiciones.** faster-whisper `small` usa las mismas opciones que large-v3-turbo en la Parte 1: español fijo, `beam_size=5` y el mismo VAD. Así la diferencia entre los dos se debe solo al tamaño del modelo. Turbo entra a la tabla como tercer punto de comparación, sin volver a ejecutarlo.
- **Alternativos con sus valores por defecto**, salvo lo que exige la máquina:
  - **EasyOCR** reduce la imagen a 1600 px solo para *detectar* las líneas. Lee el texto sobre la imagen original. Con el valor por defecto, 2560 px, un manuscrito grande pide más RAM de la que hay.
  - **Parakeet** solo acepta tramos cortos, así que corre con el VAD Silero de `onnx-asr`, que corta el audio en tramos de voz.

**Por qué estos alternativos.** Tesseract no está instalado y en Windows necesita un instalador aparte, fuera de `pip`. EasyOCR se instala con `pip` y usa una arquitectura distinta de la de PaddleOCR: el detector CRAFT y un reconocedor CRNN. Parakeet TDT 0.6B v3 (NVIDIA, CC BY 4.0) es un ASR multilingüe que cubre el español, de otra familia que Whisper (FastConformer con decodificador TDT). Su versión ONNX cuantizada en int8 corre en CPU con unos 1,5 GB de RAM. MMS-1B, con 1000 millones de parámetros (unos 4 GB en float32), no cabe cómodo en los 7 GB de RAM de la laptop, que comparte con el resto del entorno.

In [4]:
OCR_PARTE1 = json.loads((RAW / "documentos" / "ocr" / f"{muestra_docs[0]}.json").read_text(encoding="utf-8"))
ASR_PARTE1 = json.loads((RAW / "audio" / "salidas" / f"{muestra_audios[0]}.json").read_text(encoding="utf-8"))
OPCIONES_WHISPER = ASR_PARTE1["opciones"]   # las mismas de la Parte 1
CONFIG = {
    "PaddleOCR": {"rol": "OCR base", "modelo": " + ".join(OCR_PARTE1["modelos"]), "version": OCR_PARTE1["paddleocr"],
                  "opciones": OCR_PARTE1["opciones"], "origen": "salidas de 01_extraccion.ipynb (sección 3)"},
    "EasyOCR": {"rol": "OCR alternativo", "modelo": "craft_mlt_25k + latin_g2", "version": version("easyocr"),
                "opciones": {"lang_list": ["es"], "gpu": False, "canvas_size": 1600}},
    "faster-whisper small": {"rol": "ASR base", "modelo": "Systran/faster-whisper-small", "version": version("faster-whisper"),
                             "opciones": {"device": "cpu", "compute_type": "int8", **OPCIONES_WHISPER}},
    "Parakeet TDT 0.6B v3": {"rol": "ASR alternativo", "modelo": "istupakov/parakeet-tdt-0.6b-v3-onnx", "version": version("onnx-asr"),
                             "opciones": {"quantization": "int8", "vad": "silero (valores por defecto de onnx-asr)"}},
    "faster-whisper large-v3-turbo": {"rol": "ASR de la Parte 1", "modelo": ASR_PARTE1["modelo"], "version": version("faster-whisper"),
                                      "opciones": {"device": "cpu", "compute_type": "int8", **OPCIONES_WHISPER},
                                      "origen": "salidas de 01_extraccion.ipynb (sección 4)"},
    "Qwen2.5-1.5B-Instruct": {"rol": "LLM de extracción", "modelo": "Qwen/Qwen2.5-1.5B-Instruct-GGUF (q4_k_m)",
                              "version": version("llama_cpp_python"),
                              "opciones": {"n_ctx": 4096, "temperature": 0, "seed": 42, "max_tokens": 400,
                                           "max_caracteres_entrada": 8000, "salida": "JSON con esquema"}},
}
(MODELOS / "configuracion_evaluacion.json").write_text(json.dumps({"fecha": FECHA, "modelos": CONFIG}, ensure_ascii=False, indent=1),
                                                       encoding="utf-8")
pd.DataFrame(CONFIG).T[["rol", "modelo", "version"]]

,rol,modelo,version
PaddleOCR,OCR base,PP-OCRv6_medium_det + PP-OCRv6_medium_rec,3.7.0
EasyOCR,OCR alternativo,craft_mlt_25k + latin_g2,1.7.2
faster-whisper small,ASR base,Systran/faster-whisper-small,1.2.1
Parakeet TDT 0.6B v3,ASR alternativo,istupakov/parakeet-tdt-0.6b-v3-onnx,0.12.0
faster-whisper large-v3-turbo,ASR de la Parte 1,mobiuslabsgmbh/faster-whisper-large-v3-turbo,1.2.1
Qwen2.5-1.5B-Instruct,LLM de extracción,Qwen/Qwen2.5-1.5B-Instruct-GGUF (q4_k_m),0.3.2
